# Executive Summary

**Objective:** 
To establish a baseline understanding of the raw dataset's shapes, health, and limitations. This notebook is strictly exploratory and read-only; no data is modified or saved here.

**Data Flow:**
*   **Inputs:** 
    * `data\raw\internship_positions.parquet`
    * `data\external\administrative_divisions.parquet`
*   **Output:** Data Quality Assessment & Profiling Notes

**Key Operations Performed:**
1. **Structural Check [<u>[click]</u>](#1-structural-check):** 
    * Evaluated the `internship_positions` dataset (28,322 rows representing available positions, 12 columns [8 string, 4 int], 19.4 MB memory usage).
    * Evaluated the `administrative_divisions` dataset (514 rows, 4 columns [2 string, 2 int], 29.7 KB memory usage).
2. **Health Assessment [<u>[click]</u>](#2-health-assessment):** Confirmed both datasets are exceptionally clean with absolutely no missing values and no duplicates detected across any columns and rows.
3. **Descriptive Statistics [<u>[click]</u>](#3-descriptive-statistics):** 
    * **Numerical Distributions:** All numerical features (`weekly_working_day`, `requested_quota`, `approved_quota`, `applicant_count`, and `acceptance_percentage`) exhibit heavy right-skewness. Notably, `weekly_working_day` behaves as a categorical variable with only 2 unique values.
    * **Categorical Distributions:** 
        * `job_title` has 13,922 unique values, with "PENGELOLA KEGIATAN KERJA" being the most common (774 occurrences).
        * `company` has 3,401 unique values, with "Perusahaan Perseroan (Persero) Bank Negara Indonesia" being the most common (302 occurrences).
        * `job_location` has 470 unique values (regencies and cities), with "South Jakarta" being the most common (3365 occurrences).
        * `education_level` has 3 education levels, with "Bachelor's Degree" being the most common (26,851 occurrences).
        * `allowed_major` has 1,306  distinct majors, with "Management" being the most common (4,887 occurrences).

**Primary Conclusions & Required Fixes for Notebook 2:**
*   **Data Quality:** Data is robust; missing value imputation is unnecessary.
*   **Outlier & Skewness Strategy:** Realistic extreme values in quotas and applicant counts will be retained to prevent survival bias.
*   **Next Steps (Preparation):** 
    * Handle geographic merging,
    * Convert `weekly_working_day` to a categorical feature,
    * Calculate Column `acceptance_percentage`,
    * Categorize `job_title` into a new `job_category` feature,
    * Group the distinct values in `allowed_major` into broad industry categories.
    * Group numerical features into bins to make the long-tailed distributions readable,
    * Perform one-hot encoding for `education_level`, and
    * Derive a binary `allows_all_majors` flag from `job_description`.

# Setup & Imports

In [9]:
# Import libraries
import pandas as pd

from src.config import EXTERNAL_DATA_DIR, RAW_DATA_DIR

In [10]:
# Load datasets
adm_divisions = pd.read_parquet(EXTERNAL_DATA_DIR / "administrative_divisions.parquet")
internship_positions = pd.read_parquet(RAW_DATA_DIR / "internship_positions.parquet")

# 1. Structural Check

In [3]:
# Preview the internship position data
display(internship_positions.sample(10))

,job_id,published_at,job_title,company,job_location,education_level,allowed_major,job_description,weekly_working_day,requested_quota,approved_quota,applicant_count
26540,a243fc19-f11f-499a-8d22-59078f2e3fdb,2026-07-16T13:18:16+07:00,Pengolah Data dan Informasi Humas & Hukum,Balai Teknik Perkeretaapian Kelas II Padang,Kab. Padang Pariaman,Bachelor,"Ilmu Komunikasi, Hukum",1. Menyusun dan mengolah data administrasi keh...,5,1,1,21
8936,a2401225-4907-4051-9c20-19ce31bce267,2026-07-16T10:04:19+07:00,Camera Person,PT. Lativi Mediakarya (Tvone),Kota Adm. Jakarta Timur,"Diploma, Bachelor","Manajemen & Produksi Film Video & TV, Film dan...",Jobdesc :,5,2,2,11
13796,a242f01a-e711-4d1a-b624-c84b4fa4728a,2026-07-16T10:17:54+07:00,Business Process Management Intern,Jakarta Propertindo Perseroda,Kota Adm. Jakarta Pusat,"Diploma, Bachelor","Manajemen, Teknik Industri, Administrasi, Admi...",1. Menyusun dan mengembangkan materi sosialisa...,5,1,1,7
8300,a243500f-ea25-4f33-948b-6d6f5c3ad703,2026-07-16T12:43:14+07:00,Perawat,Rumah Sakit Umum Pusat Dr. M. Djamil Padang,Kota Padang,"Diploma, Profession",Keperawatan,"""1. memelihara kebersihan ruang rawat dan ling...",6,300,50,250
23298,a240db60-c7c3-4bc6-9757-b312edf36a75,2026-07-16T12:48:27+07:00,PERAWAT,RUMAH TAHANAN NEGARA KELAS IIB WONOSOBO,Kab. Wonosobo,"Bachelor, Profession","Keperawatan, Ilmu Keperawatan, Keperawatan",1. Memberikan perawatan medis dasar bagi pegaw...,6,1,1,14
23569,a241a059-b8e8-4a40-9a27-cf43e7b00398,2026-07-16T10:44:21+07:00,QUALITY SYSTEM,PT Hadji Kalla,Kota Makassar,Bachelor,"Manajemen, Teknik Industri, Statistika",1. Membantu pengumpulan data pelaksanaan Quali...,5,1,1,14
26872,a22b40bf-49dc-4af6-900c-883c293c9b8c,2026-07-16T11:20:08+07:00,Administrasi Rumah Sakit,Yayasan Rumah Sakit Islam Sumatera Barat,Kab. Pasaman Barat,"Diploma, Bachelor",Adminsitrasi Rumah Sakit,"• Orientasi RS, visi misi, budaya kerja\n• Str...",6,1,1,22
15037,a241b466-e31b-4946-81c1-e9aa54e8401c,2026-07-16T10:53:46+07:00,PETUGAS ADMINISTRASI KESEHATAN,Rs Bina Sehat Jember,Kab. Jember,"Bachelor, Diploma","Administrasi Rumah Sakit, Ilmu Kesehatan Masya...",etugas administrasi kesehatan adalah profesion...,6,9,4,31
27369,a23f5779-f05a-4493-bc32-aef15d18dd46,2026-07-16T11:52:25+07:00,Pengelola SDM,KANTOR WILAYAH DIREKTORAT JENDERAL PEMASYARAKA...,Kota Jambi,Bachelor,Manajemen,"""""""1. Mengumpulkan data dan informasi yang rel...",5,1,1,25
10664,a23101ac-0b45-4642-901b-73ddcfdaeeb9,2026-07-16T10:29:37+07:00,Qaulity Assurance Staff,PT Metindo Erasakti,Kota Bekasi,Bachelor,"Teknik Metalurgi, Teknik Industri",Bertanggung jawab dalam memastikan seluruh sis...,5,1,1,6


In [4]:
# Preview the administrative division data
display(adm_divisions.sample(10))

,regency_id,province_id,regency,province
8,1109,11,Kab. Pidie,Aceh
454,8104,81,Kab. Buru,Maluku
396,7212,72,Kab. Morowali Utara,Sulawesi Tengah
382,7172,71,Kota Bitung,Sulawesi Utara
193,3307,33,Kab. Wonosobo,Jawa Tengah
482,9111,91,Kab. Manokwari Selatan,Papua Barat
410,7313,73,Kab. Wajo,Sulawesi Selatan
387,7203,72,Kab. Morowali,Sulawesi Tengah
200,3314,33,Kab. Sragen,Jawa Tengah
368,6504,65,Kab. Nunukan,Kalimantan Utara


In [5]:
# Check the dimensions
print("Internship Positions     : ", internship_positions.shape)
print("Administrative Divisions : ", adm_divisions.shape)

Internship Positions     :  (28322, 12)
Administrative Divisions :  (514, 4)


# 2. Health Assessment

In [6]:
# Check column names, non-null counts, and data types
print("==========================")
print("Internship Positions")
print("==========================")
print(internship_positions.info())

print("\n\n==========================")
print("Administrative Divisions:")
print("==========================")
print(adm_divisions.info())

Internship Positions
<class 'pandas.DataFrame'>
RangeIndex: 28322 entries, 0 to 28321
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   job_id              28322 non-null  str  
 1   published_at        28322 non-null  str  
 2   job_title           28322 non-null  str  
 3   company             28322 non-null  str  
 4   job_location        28322 non-null  str  
 5   education_level     28322 non-null  str  
 6   allowed_major       28322 non-null  str  
 7   job_description     28322 non-null  str  
 8   weekly_working_day  28322 non-null  int64
 9   requested_quota     28322 non-null  int64
 10  approved_quota      28322 non-null  int64
 11  applicant_count     28322 non-null  int64
dtypes: int64(4), str(8)
memory usage: 19.4 MB
None


Administrative Divisions:
<class 'pandas.DataFrame'>
RangeIndex: 514 entries, 0 to 513
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  -----

In [7]:
# Check for duplicates
print(internship_positions.duplicated().sum(), " duplicates detected in Internship Positions data.")
print(adm_divisions.duplicated().sum(), " duplicates detected in Administrative Divisions data.")

0  duplicates detected in Internship Positions data.
0  duplicates detected in Administrative Divisions data.


# 3. Descriptive Statistics
## 3.1 Numerical Distributions

In [8]:
# Check the mean, min, median, and max
display(internship_positions.describe())

,weekly_working_day,requested_quota,approved_quota,applicant_count
count,28322.000000,28322.000000,28322.000000,28322.000000
mean,5.283666,1.986371,1.912224,15.385743
std,0.450785,4.696054,3.480188,19.807408
min,5.000000,1.000000,1.000000,0.000000
25%,5.000000,1.000000,1.000000,6.000000
50%,5.000000,1.000000,1.000000,10.000000
75%,6.000000,2.000000,2.000000,18.000000
max,6.000000,300.000000,150.000000,566.000000


## 3.2 Categorical Distributions

In [9]:
# See unique counts and the most frequent values
display(internship_positions.describe(include=["object", "string"]))

,job_id,published_at,job_title,company,job_location,education_level,allowed_major,job_description
count,28322,28322,28322,28322,28322,28322,28322,28322
unique,28322,3546,13922,3401,470,15,14089,17259
top,a240f2ba-12c0-4958-b416-c3e9c1d4e344,2026-07-16T12:34:14+07:00,PENGELOLA KEGIATAN KERJA,Perusahaan Perseroan (Persero) Bank Negara Ind...,Kota Adm. Jakarta Selatan,Bachelor,Psikologi,$24
freq,1,53,774,302,3365,15789,642,735


In [7]:
# Analyze all the distinct values in Column `education_level`
ed_levels = internship_positions["education_level"].str.split(", ").explode()
ed_levels = ed_levels.astype("str").str.strip()

print(f"{ed_levels.nunique()} distinct values found in Column education level.")
display(ed_levels.value_counts())

3 distinct values found in Column education level.


education_level
Bachelor      26851
Diploma       11825
Profession     2378
Name: count, dtype: int64

In [12]:
# Analyze all the distinct values in Column `allowed_major`
majors = internship_positions["allowed_major"].str.split(", ").explode()
majors = majors.astype("str").str.strip()

print(f"{majors.nunique()} distinct values found in Column allowed_major.")
display(majors.value_counts().sort_values(ascending=False).head())

1306 distinct values found in Column allowed_major.


allowed_major
Manajemen             4887
Ilmu Komunikasi       3471
Akuntansi             2725
Teknik Industri       2029
Teknik Informatika    1928
Name: count, dtype: int64